# Импорты

In [139]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Входные данные

Область определения: <br>
$\displaystyle
G = (0; 1) \times (0; 1) \\
\Gamma - \text{ граница } G \\
\overline{G} = G + \Gamma = [0; 1] \times [0; 1] \\
\overline{x} = (x; y) \in \overline{G} \\
u(x, y, t) \in \overline{Q}_T = \overline{G} \times [0; T]
$

Решение: <br>
$
u = t\sin \pi x \cdot \sin \pi y
$

Уравнение: <br>
$\displaystyle
\forall (x, y, t) \in Q_T = G \times (0; T]: \\
u'_t = Lu + f(x, y, t), ~~~ Lu = (L_x + L_y) u = \left( \frac{\partial^2}{\partial x^2} + \frac{\partial^2}{\partial y^2} \right) u \\
\forall (x, y, t) \in \Gamma \times [0; T]: \\
u(x, y, t) = \mu(x, y, t) \\
\forall (x, y) \in \overline{G}: \\
u(x, y, 0) = u_0(x, y)
$

Определение неизвестных функций: <br>
$\displaystyle
u'_t = \sin \pi x \cdot \sin \pi y \\
u''_{xx} = u''_{yy} = -\pi^2 t \sin \pi x \cdot \sin \pi y \\
f(x, y, t) = \sin \pi x \cdot \sin \pi y (1 + 2\pi^2 t) \\
\mu(x, y, t) = u_0(x, y) = 0 \\
$

In [140]:
def u(x: float, y: float, t: float) -> float:
    return t * np.sin(np.pi * x) * np.sin(np.pi * y)

def f(x: float, y: float, t: float) -> float:
    return np.sin(np.pi * x) * np.sin(np.pi * y) * (1 + 2 * np.pi ** 2 * t)

def mu(x: float, y: float, t: float) -> float:
    return 0

def u_0(x: float, y: float) -> float:
    return 0

l_x = 1
l_y = 1
T = 1

# Решение

## Монотонная прогонка

In [141]:
def monotonous_reduction(A: np.array, B: np.array, C: np.array, F: np.array) -> np.array:
    n = len(A) - 1

    alpha = np.zeros(n + 1)
    beta = np.zeros(n + 1)
    U = np.zeros(n + 1)

    alpha[1] = -C[0] / B[0]
    beta[1] = F[0] / B[0]

    for i in range(1, n):
        alpha[i + 1] = -C[i] / (B[i] + alpha[i] * A[i])
        beta[i + 1] = (F[i] - beta[i] * A[i]) / (B[i] + alpha[i] * A[i])

    U[n] = (F[n] - beta[n] * A[n]) / (B[n] + alpha[n] * A[n])

    for i in range(n - 1, -1, -1):
        U[i] = alpha[i + 1] * U[i + 1] + beta[i + 1]

    return U

## Построение сетки

$\displaystyle
\overline{\omega}_h = \left\{ \overline{x}_i = (i_x h_x, i_y h_y) ~\Big|~ i_x = \overline{0, N_x}, ~~ i_y = \overline{0, N_y}, ~~ h_x = \frac{1}{N_x}, ~~ h_y = \frac{1}{N_y} \right\} \\
\overline{\gamma}_h = \{ \overline{x}_i = (i_x h_x, i_y h_y) ~|~ i_x = 0, N_x, ~ i_y = \overline{0, N_y}; ~~ i_y = 0, N_y, ~ i_x = \overline{0, N_x} \} \\
h_x = h_y = 0.1 \\
\varphi_{i, j}^n = f(x_i, y_j, t_n) \\
\Lambda = \Lambda_x + \Lambda_y \\
\Lambda_x v_{i, j}^n = \frac{v_{i + 1, j}^n - 2v_{ij}^n + v_{i - 1, j}^n}{h_x^2} \\
\Lambda_y v_{i, j}^n = \frac{v_{i, j + 1}^n - 2v_{ij}^n + v_{i, j - 1}^n}{h_y^2} \\
$

In [142]:
N_x = 10
N_y = 10
N_t = 10

h_x = l_x / N_x
h_y = l_y / N_y
tau = T / N_t

x = np.linspace(0, l_x, N_x + 1)
y = np.linspace(0, l_y, N_y + 1)
t = np.linspace(0, T, N_t + 1)

phi = np.zeros((N_x + 1, N_y + 1, N_t + 1))
u_exact = np.zeros((N_x + 1, N_y + 1, N_t + 1))
for n in range(N_t + 1):
    for j in range(N_y + 1):
        for i in range(N_x + 1):
            phi[i, j, n] = f(x[i], y[j], t[n])
            u_exact[i, j, n] = u(x[i], y[j], t[n])

## Аппроксимация

Схема первого полушага: <br>
$\displaystyle
\frac{v_{i, j}^{n + \textstyle \frac{1}{2}} - v_{i, j}^n}{\tau} = \sigma_x \Lambda_x v_{i, j}^{n + \textstyle \frac{1}{2}} + (1 - \sigma_x) \Lambda_x v_{i, j}^n + \Lambda_y v_{i, j}^n + \varphi_{i, j}^n = \left( 0.5 - \frac{h_x}{8\tau} \right) \frac{v_{i + 1, j}^{n + \textstyle \frac{1}{2}} - 2v_{ij}^{n + \textstyle \frac{1}{2}} + v_{i - 1, j}^{n + \textstyle \frac{1}{2}}}{h_x^2} + \left( 0.5 + \frac{h_x}{8\tau} \right) \frac{v_{i + 1, j}^n - 2v_{ij}^n + v_{i - 1, j}^n}{h_x^2} + \frac{v_{i, j + 1}^n - 2v_{ij}^n + v_{i, j - 1}^n}{h_y^2} + \varphi_{i, j}^n \\
\sigma_x = 0.5 - \frac{h_x}{8\tau} > 0 \\
\text{При } x = 0; 1: \\
\overline{\mu} = \mu_{ij}^{n + 1} - 0.5\tau \Lambda_y (\mu_{ij}^{n + 1} - \mu_{ij}^n) = 0, \\
v_{0j}^{n + \textstyle \frac{1}{2}} = v_{N_x j}^{n + \textstyle \frac{1}{2}} = \overline{\mu} = 0 \\
$

СЛАУ первого полушага: <br>
$\displaystyle
\left( 1 + \frac{\tau}{h_x^2} - \frac{1}{4 h_x} \right) v_{1j}^{n + \textstyle \frac{1}{2}} + \left( \frac{1}{8 h_x} - \frac{\tau}{2 h_x^2} \right) v_{2j}^ {n + \textstyle \frac{1}{2}} = v_{1j}^n + \left( \frac{0.5\tau}{h_x^2} + \frac{1}{8 h_x} \right) (v_{2j}^n - 2 v_{1j}^n) + \frac{\tau}{h_y^2} (v_{1, j + 1}^n - 2 v_{1j}^n + v_{1, j - 1}^n) + \tau \varphi_{1j}^n, ~~ \forall j = \overline{1, N_y - 1} \\
\left( \frac{1}{8 h_x} - \frac{\tau}{2 h_x^2} \right) v_{i - 1, j}^{n + \textstyle \frac{1}{2}} + \left( 1 + \frac{\tau}{h_x^2} - \frac{1}{4 h_x} \right) v_{ij}^{n + \textstyle \frac{1}{2}} + \left( \frac{1}{8 h_x} - \frac{\tau}{2 h_x^2} \right) v_{i + 1, j}^{n + \textstyle \frac{1}{2}} = v_{ij}^n + \left( \frac{0.5\tau}{h_x^2} + \frac{1}{8 h_x} \right) (v_{i + 1, j}^n - 2 v_{ij}^n + v_{i - 1, j}^n) + \frac{\tau}{h_y^2} (v_{i, j + 1}^n - 2 v_{ij}^n + v_{i, j - 1}^n) + \tau \varphi_{ij}^n, ~~ \forall i = \overline{2, N_x - 2}, \forall j = \overline{1, N_y - 1} \\
\left( \frac{1}{8 h_x} - \frac{\tau}{2 h_x^2} \right) v_{N_x - 2, j}^ {n + \textstyle \frac{1}{2}} + \left( 1 + \frac{\tau}{h_x^2} - \frac{1}{4 h_x} \right) v_{N_x - 1, j}^{n + \textstyle \frac{1}{2}} = v_{N_X - 1, j}^n + \left( \frac{0.5\tau}{h_x^2} + \frac{1}{8 h_x} \right) (- 2 v_{N_x - 1, j}^n + v_{N_x - 2, j}^n) + \frac{\tau}{h_y^2} (v_{N_x - 1, j + 1}^n - 2 v_{N_x - 1, j}^n + v_{N_x - 1, j - 1}^n) + \tau \varphi_{N_x - 1, j}^n, ~~ \forall j = \overline{1, N_y - 1} \\
\left(\begin{array}{cccccccc}
    \displaystyle 1 + \frac{2\tau \sigma_x}{h_x^2} & \displaystyle -\frac{\tau \sigma_x}{h_x^2} & 0 & 0 & \ldots & 0 & 0 & 0 & 0 \\
    \displaystyle -\frac{\tau \sigma_x}{h_x^2} & \displaystyle 1 + \frac{2\tau \sigma_x}{h_x^2} & \displaystyle -\frac{\tau \sigma_x}{h_x^2} & 0 & \ldots & 0 & 0 & 0 & 0 \\
    0 & \displaystyle -\frac{\tau \sigma_x}{h_x^2} & \displaystyle 1 + \frac{2\tau \sigma_x}{h_x^2} & \displaystyle -\frac{\tau \sigma_x}{h_x^2} & \ldots & 0 & 0 & 0 & 0 \\
    \vdots & \vdots & \vdots & \vdots & \ddots & \vdots & \vdots & \vdots & \vdots \\
    0 & 0 & 0 & 0 & \ldots & \displaystyle \displaystyle -\frac{\tau \sigma_x}{h_x^2} & \displaystyle 1 + \frac{2\tau \sigma_x}{h_x^2} & \displaystyle -\frac{\tau \sigma_x}{h_x^2} & 0 \\
    0 & 0 & 0 & 0 & \ldots & 0 & \displaystyle -\frac{\tau \sigma_x}{h_x^2} & \displaystyle 1 + \frac{2\tau \sigma_x}{h_x^2} & \displaystyle -\frac{\tau \sigma_x}{h_x^2} \\
    0 & 0 & 0 & 0 & \ldots & 0 & 0 & \displaystyle -\frac{\tau \sigma_x}{h_x^2} & \displaystyle 1 + \frac{2\tau \sigma_x}{h_x^2} \\
\end{array}\right) \cdot \left(\begin{array}{c}
    v_{1j}^{n + \textstyle \frac{1}{2}} \\ v_{2j}^{n + \textstyle \frac{1}{2}} \\v_{3j}^{n + \textstyle \frac{1}{2}} \\ \vdots \\ v_{N_x - 3, j}^{n + \textstyle \frac{1}{2}} \\ v_{N_x - 2, j}^{n + \textstyle \frac{1}{2}} \\ v_{N_x - 1, j}^{n + \textstyle \frac{1}{2}}
\end{array}\right) = \left(\begin{array}{c}
    \displaystyle v_{1j}^n + \left( \frac{0.5\tau}{h_x^2} + \frac{1}{8 h_x} \right) (v_{2j}^n - 2 v_{1j}^n) + \frac{\tau}{h_y^2} (v_{1, j + 1}^n - 2 v_{1j}^n + v_{1, j - 1}^n) + \tau \varphi_{1j}^n \\
    \displaystyle v_{2j}^n + \left( \frac{0.5\tau}{h_x^2} + \frac{1}{8 h_x} \right) (v_{3j}^n - 2 v_{2j}^n + v_{1j}^n) + \frac{\tau}{h_y^2} (v_{2, j + 1}^n - 2 v_{2j}^n + v_{2, j - 1}^n) + \tau \varphi_{2j}^n \\
    \displaystyle v_{3j}^n + \left( \frac{0.5\tau}{h_x^2} + \frac{1}{8 h_x} \right) (v_{4j}^n - 2 v_{3j}^n + v_{2j}^n) + \frac{\tau}{h_y^2} (v_{3, j + 1}^n - 2 v_{3j}^n + v_{3, j - 1}^n) + \tau \varphi_{3j}^n \\
    \vdots \\
    \displaystyle v_{N_x - 3, j}^n + \left( \frac{0.5\tau}{h_x^2} + \frac{1}{8 h_x} \right) (v_{N_x - 2, j}^n - 2 v_{N_x - 3, j}^n + v_{N_x - 4, j}^n) + \frac{\tau}{h_y^2} (v_{N_x - 3, j + 1}^n - 2 v_{N_x - 3, j}^n + v_{N_x - 3, j - 1}^n) + \tau \varphi_{N_X - 3, j}^n \\
    \displaystyle v_{N_x - 2, j}^n + \left( \frac{0.5\tau}{h_x^2} + \frac{1}{8 h_x} \right) (v_{N_x - 1, j}^n - 2 v_{N_x - 2, j}^n + v_{N_x - 3, j}^n) + \frac{\tau}{h_y^2} (v_{N_x - 2, j + 1}^n - 2 v_{N_x - 2, j}^n + v_{N_x - 2, j - 1}^n) + \tau \varphi_{N_X - 2, j}^n \\
    \displaystyle v_{N_X - 1, j}^n + \left( \frac{0.5\tau}{h_x^2} + \frac{1}{8 h_x} \right) (- 2 v_{N_x - 1, j}^n + v_{N_x - 2, j}^n) + \frac{\tau}{h_y^2} (v_{N_x - 1, j + 1}^n - 2 v_{N_x - 1, j}^n + v_{N_x - 1, j - 1}^n) + \tau \varphi_{N_x - 1, j}^n
\end{array}\right)
$

Схема второго полушага: <br>
$\displaystyle
\frac{v_{ij}^{n + 1} - v_{ij}^{n + \textstyle \frac{1}{2}}}{0.5 \tau} = \Lambda_y (v_{ij}^{n + 1} - v_{ij}^n) = \frac{v_{i, j + 1}^{n + 1} - 2 v_{ij}^{n + 1} + v_{i, j - 1}^{n + 1}}{h_y^2} - \frac{v_{i, j + 1}^n - 2 v_{ij}^n + v_{i, j - 1}^n}{h_y^2} \\
\text{При } y = 0; 1: \\
v_{i0}^{n + 1} = v_{iN_y}^{n + 1} = \mu_{ij}^{n + 1} = 0
$

СЛАУ второго полушага: <br>
$\displaystyle
\left( 1 + \frac{\tau}{h_y^2} \right) v_{i1}^{n + 1} - \frac{\tau}{2 h_y^2} v_{i2}^{n + 1} = v_{i1}^{n + \textstyle \frac{1}{2}} - \frac{\tau}{2 h_y^2} (v_{i2}^n - 2 v_{i1}^n), ~~ \forall i = \overline{0, N_x} \\
-\frac{\tau}{2 h_y^2} v_{i, j - 1}^{n + 1} + \left( 1 + \frac{\tau}{h_y^2} \right) v_{ij}^{n + 1} - \frac{\tau}{2 h_y^2} v_{i, j + 1}^{n + 1} = v_{ij}^{n + \textstyle \frac{1}{2}} - \frac{\tau}{2 h_y^2} (v_{i, j + 1}^n - 2 v_{ij}^n + v_{i, j - 1}^n), ~~ \forall j = \overline{2, N_y - 2}, \forall i = \overline{0, N_x} \\
-\frac{\tau}{2 h_y^2} v_{i, N_y - 2}^{n + 1} + \left( 1 + \frac{\tau}{h_y^2} \right) v_{i, N_y - 1}^{n + 1} = v_{i, N_y - 1}^{n + \textstyle \frac{1}{2}} - \frac{\tau}{2 h_y^2} (- 2 v_{i, N_y - 1}^n + v_{i, N_y - 2}^n), ~~ \forall i = \overline{0, N_x} \\
\left(\begin{array}{cccccccc}
    \displaystyle 1 + \frac{\tau}{h_y^2} & \displaystyle -\frac{\tau}{2 h_y^2} & 0 & 0 & \ldots & 0 & 0 & 0 & 0 \\
    \displaystyle -\frac{\tau}{2 h_y^2} & \displaystyle 1 + \frac{\tau}{h_y^2} & \displaystyle -\frac{\tau}{2 h_y^2} & 0 & \ldots & 0 & 0 & 0 & 0 \\
    0 & \displaystyle -\frac{\tau}{2 h_y^2} & \displaystyle 1 + \frac{\tau}{h_y^2} & \displaystyle -\frac{\tau}{2 h_y^2} & \ldots & 0 & 0 & 0 & 0 \\
    \vdots & \vdots & \vdots & \vdots & \ddots & \vdots & \vdots & \vdots & \vdots \\
    0 & 0 & 0 & 0 & \ldots & \displaystyle -\frac{\tau}{2 h_y^2} & \displaystyle 1 + \frac{\tau}{h_y^2} & \displaystyle -\frac{\tau}{2 h_y^2} & 0 \\
    0 & 0 & 0 & 0 & \ldots & 0 & \displaystyle -\frac{\tau}{2 h_y^2} & \displaystyle 1 + \frac{\tau}{h_y^2} & \displaystyle -\frac{\tau}{2 h_y^2} \\
    0 & 0 & 0 & 0 & \ldots & 0 & 0 & \displaystyle -\frac{\tau}{2 h_y^2} & \displaystyle 1 + \frac{\tau}{h_y^2} \\
\end{array}\right) \cdot \left(\begin{array}{c}
    v_{i1}^{n + 1} \\ v_{i2}^{n + 1} \\ v_{i3}^{n + 1} \\ \vdots \\ v_{i, N_y - 3}^{n + 1} \\ v_{i, N_y - 2}^{n + 1} \\ v_{i, N_y - 1}^{n + 1}
\end{array}\right) = \left(\begin{array}{c}
    \displaystyle v_{i1}^{n + \textstyle \frac{1}{2}} - \frac{\tau}{2 h_y^2} (v_{i2}^n - 2 v_{i1}^n) \\
    \displaystyle v_{i2}^{n + \textstyle \frac{1}{2}} - \frac{\tau}{2 h_y^2} (v_{i3}^n - 2 v_{i2}^n + v_{i1}^n) \\
    \displaystyle v_{i3}^{n + \textstyle \frac{1}{2}} - \frac{\tau}{2 h_y^2} (v_{i4}^n - 2 v_{i3}^n + v_{i2}^n) \\
    \vdots \\
    \displaystyle v_{i, N_y - 3}^{n + \textstyle \frac{1}{2}} - \frac{\tau}{2 h_y^2} (v_{i, N_y - 2}^n - 2 v_{i, N_y - 3}^n + v_{i, N_y - 4}^n) \\
    \displaystyle v_{i, N_y - 2}^{n + \textstyle \frac{1}{2}} - \frac{\tau}{2 h_y^2} (v_{i, N_y - 1}^n - 2 v_{i, N_y - 2}^n + v_{i, N_y - 3}^n) \\
    \displaystyle v_{i, N_y - 1}^{n + \textstyle \frac{1}{2}} - \frac{\tau}{2 h_y^2} (- 2 v_{i, N_y - 1}^n + v_{i, N_y - 2}^n)
\end{array}\right)
$

In [143]:
v = np.zeros((N_x + 1, N_y + 1, N_t + 1))

sigma_x = 0.5 - h_x / (8 * tau)
print("Выполнение условия на sigma_x:", sigma_x > 0)

A_1 = np.zeros(N_x - 1)
B_1 = np.zeros(N_x - 1)
C_1 = np.zeros(N_x - 1)

A_1[1:] = -tau * sigma_x / h_x ** 2
B_1[:] = 1 + 2 * tau * sigma_x / h_x ** 2
C_1[:-1] = -tau * sigma_x / h_x ** 2

A_2 = np.zeros(N_y - 1)
B_2 = np.zeros(N_y - 1)
C_2 = np.zeros(N_y - 1)

A_2[1:] = -tau / (2 * h_y ** 2)
B_2[:] = 1 + tau / (h_y ** 2)
C_2[:-1] = -tau / (2 * h_y ** 2)

for n in range(N_t):
    v_1_2 = np.zeros((N_x + 1, N_y + 1))

    for j in range(1, N_y):
        F = v[1:-1, j, n] +\
              tau * (1 - sigma_x) * (v[2:, j, n] - 2 * v[1:-1, j, n] + v[:-2, j, n]) / h_x ** 2 +\
              tau * (v[1:-1, j + 1, n] - 2 * v[1:-1, j, n] + v[1:-1, j - 1, n]) / h_y ** 2 +\
              tau * phi[1:-1, j, n]
        v_1_2[1:-1, j] = monotonous_reduction(A_1, B_1, C_1, F)

    for i in range(1, N_x):
        F = v_1_2[i, 1:-1] - tau / (2 * h_y ** 2) * (v[i, 2:, n] - 2 * v[i, 1:-1, n] + v[i, :-2, n])
        v[i, 1:-1, n + 1] = monotonous_reduction(A_2, B_2, C_2, F)

Выполнение условия на sigma_x: True


# Визуализация

## Сравнение

### Тепловые карты

In [144]:
z_min = min(v.min(), u_exact.min())
z_max = max(v.max(), u_exact.max())

fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=("v", "u"),
    shared_yaxes=True
)

trace_numeric = go.Heatmap(
    x=x, y=y, z=v[:, :, 0].T, 
    zmin=z_min, zmax=z_max, colorscale='Viridis', colorbar=dict(x=0.45)
)
trace_exact = go.Heatmap(
    x=x, y=y, z=u_exact[:, :, 0].T, 
    zmin=z_min, zmax=z_max, colorscale='Viridis', colorbar=dict(x=1.0)
)

fig.add_trace(trace_numeric, row=1, col=1)
fig.add_trace(trace_exact, row=1, col=2)

frames = []
for k in range(len(t)):
    frames.append(
        go.Frame(
            data=[
                go.Heatmap(z=v[:, :, k].T),
                go.Heatmap(z=u_exact[:, :, k].T)
            ],
            name=f"t = {t[k]:.3f}"
        )
    )

fig.frames = frames

updatemenus = [
    {
        "buttons": [
            {
                "args": [None, {"frame": {"duration": 200, "redraw": True}, "fromcurrent": True}],
                "label": "▶ Play",
                "method": "animate"
            },
            {
                "args": [[None], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate", "transition": {"duration": 0}}],
                "label": "⏸ Pause",
                "method": "animate"
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "type": "buttons",
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }
]

sliders = [
    {
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 16},
            "prefix": "Время: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 100, "easing": "cubic-in-out"},
        "pad": {"b": 10, "t": 50},
        "len": 0.9,
        "x": 0.1,
        "y": 0,
        "steps": [
            {
                "args": [[f"t = {t[k]:.3f}"], {"frame": {"duration": 100, "redraw": True}, "mode": "immediate"}],
                "label": f"{t[k]:.3f}",
                "method": "animate"
            } for k in range(len(t))
        ]
    }
]

fig.update_layout(
    updatemenus=updatemenus,
    sliders=sliders,
    width=1100,
    height=600
)

fig.update_xaxes(title_text="x", row=1, col=1)
fig.update_xaxes(title_text="x", row=1, col=2)
fig.update_yaxes(title_text="y", row=1, col=1)

fig.show()

### Поверхности

In [145]:
z_min = min(v.min(), u_exact.min())
z_max = max(v.max(), u_exact.max())

fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=("v", "u"),
    specs=[[{"type": "surface"}, {"type": "surface"}]]
)

trace_numeric = go.Surface(
    x=x, y=y, z=v[:, :, 0].T, 
    cmin=z_min, cmax=z_max, colorscale='Viridis', colorbar=dict(x=0.45)
)
trace_exact = go.Surface(
    x=x, y=y, z=u_exact[:, :, 0].T,
    cmin=z_min, cmax=z_max, colorscale='Viridis', colorbar=dict(x=1.0)
)

fig.add_trace(trace_numeric, row=1, col=1)
fig.add_trace(trace_exact, row=1, col=2)

frames = []
for k in range(len(t)):
    frames.append(
        go.Frame(
            data=[
                go.Surface(z=v[:, :, k].T),
                go.Surface(z=u_exact[:, :, k].T)
            ],
            name=f"t = {t[k]:.3f}"
        )
    )

fig.frames = frames

updatemenus = [
    {
        "buttons": [
            {
                "args": [None, {"frame": {"duration": 200, "redraw": True}, "fromcurrent": True}],
                "label": "▶ Play",
                "method": "animate"
            },
            {
                "args": [[None], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate", "transition": {"duration": 0}}],
                "label": "⏸ Pause",
                "method": "animate"
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "type": "buttons",
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }
]

sliders = [
    {
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 16},
            "prefix": "Время: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 100, "easing": "cubic-in-out"},
        "pad": {"b": 10, "t": 50},
        "len": 0.9,
        "x": 0.1,
        "y": 0,
        "steps": [
            {
                "args": [[f"t = {t[k]:.3f}"], {"frame": {"duration": 100, "redraw": True}, "mode": "immediate"}],
                "label": f"{t[k]:.3f}",
                "method": "animate"
            } for k in range(len(t))
        ]
    }
]

scene_config = dict(
    xaxis=dict(title='x'),
    yaxis=dict(title='y'),
    zaxis=dict(title='u', range=[z_min, z_max])
)

fig.update_layout(
    updatemenus=updatemenus,
    sliders=sliders,
    width=1200,
    height=700,
    scene=scene_config,
    scene2=scene_config 
)

fig.show()

## Погрешность

In [146]:
R = np.abs(v - u_exact)

### Тепловые карты

In [147]:
z_min = R.min()
z_max = R.max()

fig = make_subplots(
    rows=1, cols=1, 
    subplot_titles=("|u - v|"),
    shared_yaxes=True
)

trace_R = go.Heatmap(
    x=x, y=y, z=R[:, :, 0].T, 
    zmin=z_min, zmax=z_max, colorscale='Viridis', colorbar=dict(x=1.0)
)
fig.add_trace(trace_R, row=1, col=1)

frames = []
for k in range(len(t)):
    frames.append(
        go.Frame(
            data=[
                go.Heatmap(z=R[:, :, k].T)
            ],
            name=f"t = {t[k]:.3f}"
        )
    )
fig.frames = frames

updatemenus = [
    {
        "buttons": [
            {
                "args": [None, {"frame": {"duration": 200, "redraw": True}, "fromcurrent": True}],
                "label": "▶ Play",
                "method": "animate"
            },
            {
                "args": [[None], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate", "transition": {"duration": 0}}],
                "label": "⏸ Pause",
                "method": "animate"
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "type": "buttons",
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }
]

sliders = [
    {
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 16},
            "prefix": "Время: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 100, "easing": "cubic-in-out"},
        "pad": {"b": 10, "t": 50},
        "len": 0.9,
        "x": 0.1,
        "y": 0,
        "steps": [
            {
                "args": [[f"t = {t[k]:.3f}"], {"frame": {"duration": 100, "redraw": True}, "mode": "immediate"}],
                "label": f"{t[k]:.3f}",
                "method": "animate"
            } for k in range(len(t))
        ]
    }
]

fig.update_layout(
    updatemenus=updatemenus,
    sliders=sliders,
    width=900,
    height=600
)

fig.update_xaxes(title_text="x", row=1, col=1)
fig.update_yaxes(
    title_text="y", 
    row=1, col=1,
    scaleanchor="x",
    scaleratio=1
)

fig.show()

### Поверхности

In [148]:
z_min = R.min()
z_max = R.max()

fig = make_subplots(
    rows=1, cols=1, 
    subplot_titles=("|u - v|"),
    specs=[[{"type": "surface"}]]
)

trace_R = go.Surface(
    x=x, y=y, z=R[:, :, 0].T,
    cmin=z_min, cmax=z_max, colorscale='Viridis', colorbar=dict(x=1.0)
)
fig.add_trace(trace_R, row=1, col=1)

frames = []
for k in range(len(t)):
    frames.append(
        go.Frame(
            data=[
                go.Surface(z=R[:, :, k].T),
            ],
            name=f"t = {t[k]:.3f}"
        )
    )
fig.frames = frames

updatemenus = [
    {
        "buttons": [
            {
                "args": [None, {"frame": {"duration": 200, "redraw": True}, "fromcurrent": True}],
                "label": "▶ Play",
                "method": "animate"
            },
            {
                "args": [[None], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate", "transition": {"duration": 0}}],
                "label": "⏸ Pause",
                "method": "animate"
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "type": "buttons",
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }
]

sliders = [
    {
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 16},
            "prefix": "Время: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 100, "easing": "cubic-in-out"},
        "pad": {"b": 10, "t": 50},
        "len": 0.9,
        "x": 0.1,
        "y": 0,
        "steps": [
            {
                "args": [[f"t = {t[k]:.3f}"], {"frame": {"duration": 100, "redraw": True}, "mode": "immediate"}],
                "label": f"{t[k]:.3f}",
                "method": "animate"
            } for k in range(len(t))
        ]
    }
]

scene_config = dict(
    xaxis=dict(title='x'),
    yaxis=dict(title='y'),
    zaxis=dict(title='u', range=[z_min, z_max])
)

fig.update_layout(
    updatemenus=updatemenus,
    sliders=sliders,
    width=1200,
    height=700,
    scene=scene_config,
    scene2=scene_config 
)

fig.show()